In [1]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.cloud import storage

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, StructType, StructField, IntegerType, StringType, DateType

from utils.LocationFunctions import load_locations_df, get_locations_from_bq, get_missing_locations, get_batch_geocode, update_locations_bq
from utils.Common import gcs_file_read, gcs_upload_parquet, partial_parse_raw_data

gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

load_dotenv()

ScrapedRawSchema = StructType([
    StructField('content', StringType(), True),
    StructField('tweetlinkid', StringType(), True),
    StructField('created_at', DateType(), True),
])

KaggleRawSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

def initialize(app_name, data_source='scraped'):
    """Initialize Spark, GCS client, and location references for the transform pipeline.

    Args:
        app_name: Name for the Spark application.
        data_source: Data source type ('scraped' or 'kaggle').

    Returns:
        tuple: spark, gcs_client, project_id, dataset, locations_table_id,
               staging_locations_table_id, bucket_name, raw_folder,
               clean_folder, scrape_folder, df_locations
    """
    spark = SparkSession.builder \
            .master("local[*]") \
            .appName(app_name) \
            .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
            .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
            .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
            .getOrCreate()
    
    gcs_client = storage.Client()

    project_id = os.getenv("PROJECT_ID")
    dataset = os.getenv("DATASET")
    locations_table_id = f"{project_id}:{dataset}.dim_locations"
    staging_locations_table_id = f"{project_id}:{dataset}.staging_locations"
    bucket_name = os.getenv('BUCKET_NAME')
    raw_folder = os.getenv('RAW_FOLDER_NAME')
    clean_folder = os.getenv('CLEANED_FOLDER_NAME')
    scrape_folder = f"{raw_folder}/scrape"

    df_locations = load_locations_df(spark, locations_table_id)

    return spark, gcs_client, project_id, dataset, locations_table_id, staging_locations_table_id, bucket_name, raw_folder, clean_folder, scrape_folder, df_locations


In [2]:
spark, gcs_client, project_id, dataset, locations_table_id, staging_locations_table_id, bucket_name, raw_folder, clean_folder, scrape_folder, df_locations = initialize('Transform Stage (Kaggle)', 'kaggle')

kaggle_folder = f"{raw_folder}/kaggle"
raw_filename = f"{kaggle_folder}/kaggle_historical_data.csv"

bucket = gcs_client.bucket(bucket_name)
    
blob = storage.Blob(bucket=bucket, name=raw_filename)
if not blob.exists():
    print(f"Skipping (not found): {raw_filename}")

print(f"Processing: {raw_filename}")
df_raw = gcs_file_read(spark, bucket_name, raw_filename, KaggleRawSchema)
df_raw_specific = df_raw.select("Tweet", "Date", "Source")
df_raw_renamed = df_raw_specific.withColumnsRenamed({"Tweet": "content", "Date": "created_at", "Source": "tweetlinkid"})

26/06/12 06:14:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Processing: raw/kaggle/kaggle_historical_data.csv


In [ ]:
df_raw_renamed.count()

In [3]:
df_raw_renamed.printSchema()

root
 |-- content: string (nullable = true)
 |-- created_at: date (nullable = true)
 |-- tweetlinkid: string (nullable = true)



In [4]:
df_partial_parsed = partial_parse_raw_data(df_raw_renamed)

In [5]:
df_partial_parsed.printSchema()

root
 |-- date: date (nullable = true)
 |-- time: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- location: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- type: string (nullable = true)
 |-- lanes_blocked: integer (nullable = true)
 |-- involved: string (nullable = true)
 |-- post: string (nullable = true)
 |-- link: string (nullable = true)



In [9]:
null_timestamp_count = df_partial_parsed.filter(F.col("event_timestamp").isNull()).count()

26/06/12 06:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 06:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 06:16:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 06:16:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [10]:
null_timestamp_count

0

In [8]:
if null_timestamp_count > 0:
    window = Window.rowsBetween(Window.unboundedPreceding, 0)
    df_partial_parsed = df_partial_parsed.withColumn("time", F.last("time", ignorenulls=True).over(window))
    df_partial_parsed = df_partial_parsed.withColumn(
        "event_timestamp", 
        F.to_timestamp(F.concat_ws(' ', F.col("date"), F.col("time")), "yyyy-MM-dd HH:mm")
    )

In [11]:
df_full_parsed = get_locations_from_bq(df_locations, df_partial_parsed)

In [12]:
missing_locations = get_missing_locations(df_full_parsed)

In [14]:
len(missing_locations)

0